In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==============================================================================
# PHASE 1A: ENVIRONMENT SETUP & DINOv2 INITIALIZATION
# ==============================================================================
!pip install gdown optuna scikit-learn -q

import os
import random
import shutil
import numpy as np
import torch
import gdown
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

# Verify the target directory exists or create it
TARGET_DIR = "/content/drive/MyDrive/CAMO"
os.makedirs(TARGET_DIR, exist_ok=True)
print(f"✅ Google Drive successfully mounted. Target workspace ready at: {TARGET_DIR}")

# --- 1. DETERMINISTIC SEED LOCK ---
# Crucial for reproducing the exact Optuna clusters and DB scores
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# --- 2. HARDWARE ACCELERATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Active Device: {device.type.upper()}")

# --- 3. DATASET INGESTION & EXTRACTION ---
# Replace in DRIVE_FILE_ID with the actual ID from your Google Drive share link
DRIVE_FILE_ID = '1NAfyqXkxYatSkfoBwGRe3CW-rI9YfaIk'
WORKSPACE_DIR = "/content/IMAGE_WORKSPACE"
ZIP_DEST = "/content/dataset.zip"

# Clean slate if re-running
if os.path.exists(WORKSPACE_DIR):
    shutil.rmtree(WORKSPACE_DIR)
os.makedirs(WORKSPACE_DIR, exist_ok=True)

print("--- 📦 DOWNLOADING & EXTRACTING DATASET ---")
# Download directly from Drive key
gdown.download(id=DRIVE_FILE_ID, output=ZIP_DEST, quiet=False)

# Extract silently to avoid notebook output spam
!unzip -o -q {ZIP_DEST} -d {WORKSPACE_DIR}
os.remove(ZIP_DEST) # Free up disk space

# Ensure we can find the images dynamically
try:
    from glob import glob
    # Adjust this path structure depending on how your zip is nested
    img_paths = sorted(glob(f"{WORKSPACE_DIR}/**/image/*.jpg", recursive=True) + \
                       glob(f"{WORKSPACE_DIR}/**/image/*.png", recursive=True))
    print(f"✅ Successfully located {len(img_paths)} ground-truth images.")
except IndexError:
    print("❌ Error: Could not locate 'image' directory. Check zip structure.")

# --- 4. DINOv2 INITIALIZATION ---
print("--- 🧠 LOADING MACRO-SEMANTIC BACKBONE ---")
# Using vits14 for optimal speed-to-accuracy ratio in patch extraction
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
dino.eval() # Lock gradients
print("✅ DINOv2 Initialized and locked in evaluation mode.")

# Tune these based on your Colab instance
BATCH_SIZE = 256
NUM_WORKERS = 12 #For GPU
MAX_WORKERS = 32 #For CPU

✅ Google Drive successfully mounted. Target workspace ready at: /content/drive/MyDrive/CAMO
✅ Active Device: CUDA
--- 📦 DOWNLOADING & EXTRACTING DATASET ---


Downloading...
From (original): https://drive.google.com/uc?id=1NAfyqXkxYatSkfoBwGRe3CW-rI9YfaIk
From (redirected): https://drive.google.com/uc?id=1NAfyqXkxYatSkfoBwGRe3CW-rI9YfaIk&confirm=t&uuid=3db776e0-03a9-445c-b8fb-3f62c4182393
To: /content/dataset.zip

  0%|          | 0.00/2.38G [00:00<?, ?B/s]
  1%|▏         | 30.4M/2.38G [00:00<00:07, 301MB/s]
  3%|▎         | 60.8M/2.38G [00:00<00:09, 253MB/s]
  4%|▍         | 93.8M/2.38G [00:00<00:08, 283MB/s]
  5%|▌         | 123M/2.38G [00:00<00:07, 285MB/s] 
  6%|▋         | 155M/2.38G [00:00<00:07, 294MB/s]
  8%|▊         | 188M/2.38G [00:00<00:07, 306MB/s]
  9%|▉         | 224M/2.38G [00:00<00:06, 321MB/s]
 11%|█         | 256M/2.38G [00:00<00:07, 304MB/s]
 12%|█▏        | 292M/2.38G [00:00<00:06, 317MB/s]
 14%|█▎        | 326M/2.38G [00:01<00:06, 323MB/s]
 15%|█▌        | 359M/2.38G [00:01<00:11, 173MB/s]
 17%|█▋        | 395M/2.38G [00:01<00:09, 207MB/s]
 18%|█▊        | 430M/2.38G [00:01<00:08, 238MB/s]
 20%|█▉        | 469M/2.38G [0

✅ Successfully located 10000 ground-truth images.
--- 🧠 LOADING MACRO-SEMANTIC BACKBONE ---
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth



  0%|          | 0.00/84.2M [00:00<?, ?B/s]
 39%|███▉      | 32.6M/84.2M [00:00<00:00, 342MB/s]
100%|██████████| 84.2M/84.2M [00:00<00:00, 413MB/s]


✅ DINOv2 Initialized and locked in evaluation mode.


In [ ]:
# ==============================================================================
# CELL 2: GLOBAL SEMANTIC EMBEDDING & COMPRESSION
# ==============================================================================
import cv2
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

print("--- 🔍 MAPPING SEMANTIC POSSIBILITY SPACE ---")

class RawSemanticDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = cv2.imread(path)
        if img is None:
            return None

        # Convert to RGB and resize for DINOv2 (224x224)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_res = cv2.resize(img_rgb, (224, 224))

        # Normalize to [0, 1] and permute to (C, H, W)
        img_pt = torch.from_numpy(img_res).permute(2, 0, 1).float() / 255.0
        return img_pt, path

def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

# Locate raw images (Handling nested structures in the ZIP)
img_files = [path for path in img_paths if '/train/camo/image/' in path]

print(f"Targeting {len(img_files)} source images for semantic profiling...")

dataset = RawSemanticDataset(img_files)
# Batch size 64 is usually safe for T4 GPUs with DINOv2-ViT-S
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

all_feats = {"dino": [], "paths": []}

with torch.no_grad():
    for img_pt, paths in tqdm(dataloader, desc="Extracting DINOv2 Embeddings"):
        img_pt = img_pt.to(device, non_blocking=True)
        # Extract CLS token representations
        dino_out = dino(img_pt).cpu().numpy()
        all_feats["dino"].append(dino_out)
        all_feats["paths"].extend(paths)

# Stack and Normalize
V_DINO = np.vstack(all_feats["dino"])
dino_l2 = normalize(V_DINO, norm='l2', axis=1)
v_dino_scaled = StandardScaler().fit_transform(dino_l2)

# Compress feature space mathematically
pca_dino = PCA(n_components=0.95, random_state=SEED)
V_FINAL = pca_dino.fit_transform(v_dino_scaled)

del dino
torch.cuda.empty_cache()
gc.collect()

print(f"\n✅ Extraction Complete.")
print(f"✅ Original Dimensions: 384 | Compressed Dimensions (95% variance): {V_FINAL.shape[1]}")

--- 🔍 MAPPING SEMANTIC POSSIBILITY SPACE ---
Targeting 3040 source images for semantic profiling...



Extracting DINOv2 Embeddings: 100%|██████████| 12/12 [00:08<00:00,  1.35it/s]



✅ Extraction Complete.
✅ Original Dimensions: 384 | Compressed Dimensions (95% variance): 253


In [ ]:
# ==============================================================================
# CELL 3: OPTIMAL HABITAT PARTITIONING (THE CSP ENGINE)
# ==============================================================================
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import davies_bouldin_score
import optuna
import numpy as np

print("--- ⚙️ OPTIMIZING HABITAT BOUNDARIES ---")

# 🚨 THE FIX: Strip away microscopic GPU floating-point noise to lock Optuna's trajectory
V_STABLE = np.round(V_FINAL, decimals=5)

# Compute the linkage matrix ONCE using the stabilized embeddings
print("Computing Ward's Linkage Matrix...")
Z_matrix = linkage(V_STABLE, method='ward')

def objective(trial):
    # Propose a number of clusters (habitats) between 5 and 40
    k = trial.suggest_int("k", 5, 40)

    # Extract flat clusters based on k
    labels = fcluster(Z_matrix, t=k, criterion='maxclust')

    if len(np.unique(labels)) < 2:
        return float('inf') # Prevent invalid singular clusters

    # MINIMIZE Davies-Bouldin Index (Lower = better separation)
    return davies_bouldin_score(V_STABLE, labels)

# Run Optuna Study
# The TPESampler is seeded, and because V_STABLE is locked, the DB scores will be locked.
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=150, show_progress_bar=True)

optimal_k = study.best_params['k']
print(f"\n✅ CSP Optimization Complete.")
print(f"✅ Optimal Number of Habitats (k): {optimal_k}")
print(f"✅ Best Davies-Bouldin Score: {study.best_value:.4f}")

# Extract final labels using the optimal k
final_labels = fcluster(Z_matrix, t=optimal_k, criterion='maxclust')

# Print Distribution
unique_ids, counts = np.unique(final_labels, return_counts=True)
print("\n--- HABITAT DISTRIBUTION ---")
for uid, count in zip(unique_ids, counts):
    print(f"Habitat {uid:02d}: {count} images")

# --- CENTROID EXPORT FOR PHASE 2 ---
print("\n--- 🎯 COMPUTING & SAVING MACRO-ENVIRONMENT CENTROIDS ---")
habitat_centroids = {}
for uid in unique_ids:
    cluster_mask = (final_labels == uid)

    # We use dino_l2 (the uncompressed embeddings from Cell 2) to serve as anchors for Cell 7
    cluster_feats = dino_l2[cluster_mask]
    habitat_centroids[f"cluster_{uid}"] = np.mean(cluster_feats, axis=0)

CENTROID_PATH = "/content/drive/MyDrive/CAMO/habitat_centroids.npy"
np.save(CENTROID_PATH, habitat_centroids)
print(f"✅ Centroids saved successfully to {CENTROID_PATH}")

--- ⚙️ OPTIMIZING HABITAT BOUNDARIES ---
Computing Ward's Linkage Matrix...


[I 2026-05-28 01:48:59,966] A new study created in memory with name: no-name-088b1ccd-1466-40ce-8d96-93558937084b


  0%|          | 0/150 [00:00<?, ?it/s]

 57%|█████▋    | 1.36G/2.38G [02:00<00:11, 88.7MB/s]

[I 2026-05-28 01:49:00,006] Trial 0 finished with value: 3.663387462893878 and parameters: {'k': 18}. Best is trial 0 with value: 3.663387462893878.
[I 2026-05-28 01:49:00,034] Trial 1 finished with value: 3.493266986922039 and parameters: {'k': 39}. Best is trial 1 with value: 3.493266986922039.
[I 2026-05-28 01:49:00,059] Trial 2 finished with value: 3.5560871833708823 and parameters: {'k': 31}. Best is trial 1 with value: 3.493266986922039.
[I 2026-05-28 01:49:00,080] Trial 3 finished with value: 3.544543366899265 and parameters: {'k': 26}. Best is trial 1 with value: 3.493266986922039.
[I 2026-05-28 01:49:00,097] Trial 4 finished with value: 4.122498692036642 and parameters: {'k': 10}. Best is trial 1 with value: 3.493266986922039.
[I 2026-05-28 01:49:00,110] Trial 5 finished with value: 4.122498692036642 and parameters: {'k': 10}. Best is trial 1 with value: 3.493266986922039.
[I 2026-05-28 01:49:00,122] Trial 6 finished with value: 4.045909383665131 and parameters: {'k': 7}. Best

...

[I 2026-05-28 01:49:03,228] Trial 125 finished with value: 3.544543366899265 and parameters: {'k': 26}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,250] Trial 126 finished with value: 3.4879410412456657 and parameters: {'k': 24}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,273] Trial 127 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,298] Trial 128 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,326] Trial 129 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,352] Trial 130 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,374] Trial 131 finished with value: 3.481588148158041 and parame

 57%|█████▋    | 1.36G/2.38G [02:03<00:11, 88.7MB/s]

[I 2026-05-28 01:49:03,450] Trial 134 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,479] Trial 135 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,506] Trial 136 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,531] Trial 137 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,562] Trial 138 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,585] Trial 139 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,609] Trial 140 finished with value: 3.593061854178179 and paramet

 57%|█████▋    | 1.36G/2.38G [02:03<00:11, 88.7MB/s]

[I 2026-05-28 01:49:03,655] Trial 142 finished with value: 3.481588148158041 and parameters: {'k': 25}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,679] Trial 143 finished with value: 3.572060098789632 and parameters: {'k': 28}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,703] Trial 144 finished with value: 3.5443116127472774 and parameters: {'k': 22}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,731] Trial 145 finished with value: 3.593061854178179 and parameters: {'k': 27}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,758] Trial 146 finished with value: 3.4869353900277567 and parameters: {'k': 23}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,787] Trial 147 finished with value: 3.544543366899265 and parameters: {'k': 26}. Best is trial 44 with value: 3.481588148158041.
[I 2026-05-28 01:49:03,810] Trial 148 finished with value: 3.481588148158041 and param

In [ ]:
# ==============================================================================
# CELL 4 (UPDATED): DATASET ROUTING & REORGANIZATION (IMAGE, LABEL, MASK)
# ==============================================================================
import zipfile

# ⚠️ OPTIMIZATION: Write to local Colab SSD first, not directly to Google Drive
LOCAL_OUTPUT_DIR = "/content/CSP_Partitioned_Dataset"
DRIVE_DESTINATION = "/content/drive/MyDrive/CAMO/"

print(f"--- 📂 MULTI-THREADED ROUTING FILES TO {optimal_k} SEMANTIC CLUSTERS ---")

# Clean old outputs if they exist
if os.path.exists(LOCAL_OUTPUT_DIR):
    shutil.rmtree(LOCAL_OUTPUT_DIR)

# Create directory structure upfront for Images, Labels, AND Masks
for uid in unique_ids:
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, "images", f"cluster_{uid}"), exist_ok=True)
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, "labels", f"cluster_{uid}"), exist_ok=True)
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, "masks", f"cluster_{uid}"), exist_ok=True)

# Define the atomic copy task for a triplet (Image, Label, Mask)
def route_file_group(path, label):
    # Target directories
    img_out_dir = os.path.join(LOCAL_OUTPUT_DIR, "images", f"cluster_{label}")
    lbl_out_dir = os.path.join(LOCAL_OUTPUT_DIR, "labels", f"cluster_{label}")
    msk_out_dir = os.path.join(LOCAL_OUTPUT_DIR, "masks", f"cluster_{label}")

    # 1. Copy Image (Always exists if it made it to this list)
    shutil.copy(path, img_out_dir)

    # Robust path calculation for sibling directories
    # e.g., .../train/camo/image -> .../train/camo
    parent_dir = os.path.dirname(os.path.dirname(path))
    base_name = os.path.splitext(os.path.basename(path))[0]

    label_source_path = os.path.join(parent_dir, "label", base_name + ".txt")
    mask_source_path = os.path.join(parent_dir, "mask", base_name + ".png")

    # Track missing files
    miss_lbl, miss_msk = 0, 0

    # 2. Copy Label
    if os.path.exists(label_source_path):
        shutil.copy(label_source_path, lbl_out_dir)
    else:
        miss_lbl = 1

    # 3. Copy Mask
    if os.path.exists(mask_source_path):
        shutil.copy(mask_source_path, msk_out_dir)
    else:
        miss_msk = 1

    return miss_lbl, miss_msk

# Execute the copying concurrently
missing_labels = 0
missing_masks = 0
tasks = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for path, label in zip(all_feats['paths'], final_labels):
        tasks.append(executor.submit(route_file_group, path, label))

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Parallel Routing Files"):
        lbl_miss, msk_miss = future.result()
        missing_labels += lbl_miss
        missing_masks += msk_miss

print(f"\n🚀 Routing Complete! Compressing dataset for transfer to Google Drive...")

# Zip the perfectly structured folder
ZIP_NAME = "/content/CSP_Partitioned_Dataset.zip"
if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

# Using system zip for maximum speed
!zip -r -q {ZIP_NAME} {LOCAL_OUTPUT_DIR}

print(f"📦 Copying compressed dataset to Google Drive ({DRIVE_DESTINATION})...")
shutil.copy(ZIP_NAME, DRIVE_DESTINATION)

print(f"\n✅ Phase 1A Complete! Dataset successfully semantically partitioned.")
if missing_labels > 0 or missing_masks > 0:
    print(f"⚠️ Note: Missing {missing_labels} labels and {missing_masks} masks (Empty Environments).")

--- 📂 MULTI-THREADED ROUTING FILES TO 25 SEMANTIC CLUSTERS ---



Parallel Routing Files: 100%|██████████| 3040/3040 [00:02<00:00, 1108.32it/s]



🚀 Routing Complete! Compressing dataset for transfer to Google Drive...
📦 Copying compressed dataset to Google Drive (/content/drive/MyDrive/CAMO/)...

✅ Phase 1A Complete! Dataset successfully semantically partitioned.


In [ ]:
# ==============================================================================
# CELL 6: PHASE 1B - DUAL-THREADED LATENT AFFORDANCE DICTIONARY
# ==============================================================================
import os
import cv2
import json
import torch
import numpy as np
import random
import gc
from glob import glob
from tqdm import tqdm
from skimage.morphology import skeletonize
from sklearn.ensemble import IsolationForest
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- 🏗️ BUILDING CONTEXT-SYNCHRONIZED LATENT DICTIONARY ---")

# --- 1. CONFIGURATION & PATH RESOLUTION ---
sample_path = glob("/content/IMAGE_WORKSPACE/**/train", recursive=True)[0]
INPUT_YOLO_DIR = os.path.dirname(sample_path)

PARTITIONED_ROOT = "/content/CSP_Partitioned_Dataset"
CLUSTER_IMG_ROOT = os.path.join(PARTITIONED_ROOT, "images")
CLUSTER_LBL_ROOT = os.path.join(PARTITIONED_ROOT, "labels")
CLUSTER_MSK_ROOT = os.path.join(PARTITIONED_ROOT, "masks")
DICT_PATH = "/content/drive/MyDrive/CAMO/latent_affordance_dictionary.json"

print(f"✅ Local Root Synchronized at: {INPUT_YOLO_DIR}")

# --- 2. BACKBONE INITIALIZATION ---
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device).eval()
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device).eval()

dino_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 3. MATHEMATICAL EXTRACTION FUNCTIONS ---
def get_topographical_features(depth_map, bbox, w, h):
    grad_x = cv2.Sobel(depth_map, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(depth_map, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(grad_x**2 + grad_y**2 + 1.0)
    norm_x = -grad_x / magnitude
    norm_y = -grad_y / magnitude
    norm_z = 1.0 / magnitude

    _, x_c, y_c, bw, bh = bbox
    x1, x2 = max(0, int((x_c - bw/2) * w)), min(w, int((x_c + bw/2) * w))
    y1, y2 = max(0, int((y_c - bh/2) * h)), min(h, int((y_c + bh/2) * h))

    region_gx = grad_x[y1:y2, x1:x2]
    region_gy = grad_y[y1:y2, x1:x2]
    region_nx = norm_x[y1:y2, x1:x2]
    region_ny = norm_y[y1:y2, x1:x2]
    region_nz = norm_z[y1:y2, x1:x2]

    if region_gx.size == 0: return [0.0, 0.0], [0.0, 0.0, 1.0]

    grad_D = [float(np.mean(region_gx)), float(np.mean(region_gy))]
    raw_mean_normal = [float(np.mean(region_nx)), float(np.mean(region_ny)), float(np.mean(region_nz))]

    n_mag = np.sqrt(sum(n**2 for n in raw_mean_normal)) + 1e-8
    surface_normal = [n / n_mag for n in raw_mean_normal]

    return grad_D, surface_normal

def get_spherical_harmonics(gray_img, bbox, w, h, mask=None):
    """Approximates first 9 Spherical Harmonics coefficients, strictly utilizing biological mask pixels."""
    _, x_c, y_c, bw, bh = bbox
    x1, x2 = max(0, int((x_c - (bw*1.5)/2) * w)), min(w, int((x_c + (bw*1.5)/2) * w))
    y1, y2 = max(0, int((y_c - (bh*1.5)/2) * h)), min(h, int((y_c + (bh*1.5)/2) * h))
    patch = gray_img[y1:y2, x1:x2].copy()

    if patch.size == 0: return [0.0] * 9

    if mask is not None:
        patch_mask = mask[y1:y2, x1:x2]
        if np.count_nonzero(patch_mask) > 0:
            patch = patch * (patch_mask.astype(np.float32) / 255.0)

    ph, pw = patch.shape
    y_grid, x_grid = np.mgrid[-1:1:ph*1j, -1:1:pw*1j]
    r2 = x_grid**2 + y_grid**2
    valid = r2 <= 1.0
    z_grid = np.zeros_like(x_grid)
    z_grid[valid] = np.sqrt(1 - r2[valid])

    Y = [
        np.ones_like(x_grid) * 0.282095, y_grid * 0.488603, z_grid * 0.488603, x_grid * 0.488603,
        (x_grid * y_grid) * 1.092548, (y_grid * z_grid) * 1.092548,
        (3.0 * z_grid**2 - 1.0) * 0.315392, (x_grid * z_grid) * 1.092548,
        (x_grid**2 - y_grid**2) * 0.546274
    ]
    sum_valid = np.sum(valid)
    if sum_valid == 0: return [0.0] * 9
    return [float(np.sum(patch[valid] * basis[valid]) / sum_valid) for basis in Y]

def get_adaptive_solidity(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return 0.50
    main_contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(main_contour)
    if area < 50: return 0.50

    hull = cv2.convexHull(main_contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0.50

    skeleton = skeletonize(mask > 0)
    skeleton_length = np.count_nonzero(skeleton)
    average_thickness = area / skeleton_length if skeleton_length > 0 else 10.0

    if average_thickness < 5.0:
        solidity = min(1.0, solidity + (5.0 - average_thickness) * 0.1)
    return float(solidity)

# --- 4. MULTI-THREADED DATASETS ---
class EnvDataset(Dataset):
    def __init__(self, registry, transform):
        self.img_paths = list(registry.keys())
        self.transform = transform
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img = cv2.imread(path)
        if img is None: return torch.zeros(3, 256, 256), path, False, 0, 0
        h, w = img.shape[:2]

        # 🚨 FIX: Force uniform dimensions so the DataLoader can batch safely
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (256, 256))

        tensor = self.transform(img_rgb).squeeze(0)
        return tensor, path, True, h, w

class PatchDataset(Dataset):
    def __init__(self, patch_registry, transform):
        self.patch_registry = patch_registry
        self.transform = transform
    def __len__(self): return len(self.patch_registry)
    def __getitem__(self, idx):
        data = self.patch_registry[idx]
        return self.transform(data['patch']), data['obj_id']

# --- 5. BUILD IMAGE REGISTRY ---
image_registry = {}
cluster_dirs = sorted([d for d in os.listdir(CLUSTER_IMG_ROOT) if d.startswith("cluster_")])

for cluster_name in cluster_dirs:
    img_cluster_path = os.path.join(CLUSTER_IMG_ROOT, cluster_name)
    lbl_cluster_path = os.path.join(CLUSTER_LBL_ROOT, cluster_name)
    msk_cluster_path = os.path.join(CLUSTER_MSK_ROOT, cluster_name)

    for img_name in sorted(os.listdir(img_cluster_path)):
        if not img_name.endswith(('.jpg', '.png')): continue
        base_name = os.path.splitext(img_name)[0]
        img_path = os.path.join(img_cluster_path, img_name)
        lbl_path = os.path.join(lbl_cluster_path, base_name + ".txt")
        if not os.path.exists(lbl_path): continue

        true_mask_path = None
        for ext in ['.png', '.jpg', '.jpeg']:
            test_path = os.path.join(msk_cluster_path, base_name + ext)
            if os.path.exists(test_path):
                true_mask_path = test_path
                break

        with open(lbl_path, 'r') as f: lines = f.readlines()
        valid_bboxes = [[float(x) for x in line.strip().split()][:5] for line in lines if len(line.strip().split()) >= 5]

        if valid_bboxes:
            image_registry[img_path] = {'cluster': cluster_name, 'bboxes': valid_bboxes, 'mask_path': true_mask_path, 'img_name': img_name}

# --- 6. PHASE A: MIDAS BATCHING (RUNS ONCE PER IMAGE) ---
env_loader = DataLoader(EnvDataset(image_registry, midas_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

patch_registry = []
obj_metadata = {}
obj_counter = 0

with torch.no_grad():
    for batch_tensors, batch_paths, batch_valid, batch_h, batch_w in tqdm(env_loader, desc="Topography Extraction"):
        batch_tensors = batch_tensors.to(device)
        depth_preds = midas(batch_tensors)

        for i in range(len(batch_paths)):
            if not batch_valid[i]: continue
            img_path = batch_paths[i]
            H, W = batch_h[i].item(), batch_w[i].item()
            reg_info = image_registry[img_path]

            depth_map = torch.nn.functional.interpolate(
                depth_preds[i].unsqueeze(0).unsqueeze(0), size=(H, W), mode="bicubic", align_corners=False
            ).squeeze().cpu().numpy()
            depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)

            img = cv2.imread(img_path)
            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

            if reg_info['mask_path']:
                full_mask = cv2.imread(reg_info['mask_path'], cv2.IMREAD_GRAYSCALE)
                _, full_mask = cv2.threshold(full_mask, 127, 255, cv2.THRESH_BINARY)
                if full_mask.shape[:2] != (H, W): full_mask = cv2.resize(full_mask, (W, H), interpolation=cv2.INTER_NEAREST)
            else:
                full_mask = np.zeros((H, W), dtype=np.uint8)

            for bbox in reg_info['bboxes']:
                class_id = int(bbox[0])
                _, x_c, y_c, bw, bh = bbox
                area = bw * bh
                x1, x2 = max(0, int((x_c - bw/2) * W)), min(W, int((x_c + bw/2) * W))
                y1, y2 = max(0, int((y_c - bh/2) * H)), min(H, int((y_c + bh/2) * H))

                if x2 <= x1 or y2 <= y1: continue

                grad_D, surface_normal = get_topographical_features(depth_map, bbox, W, H)
                c_sh = get_spherical_harmonics(gray_img, bbox, W, H, mask=full_mask)

                obj_mask = np.zeros((H, W), dtype=np.uint8)
                if reg_info['mask_path']: obj_mask[y1:y2, x1:x2] = full_mask[y1:y2, x1:x2]
                else: cv2.rectangle(obj_mask, (x1, y1), (x2, y2), 255, -1)

                solidity = get_adaptive_solidity(obj_mask[y1:y2, x1:x2])
                pure_obj = cv2.bitwise_and(img, img, mask=obj_mask)
                cropped = pure_obj[y1:y2, x1:x2]

                if cropped.size > 0 and cropped.shape[0] > 10 and cropped.shape[1] > 10:
                    obj_id = f"{reg_info['img_name']}_obj{obj_counter}"
                    patch_registry.append({'obj_id': obj_id, 'patch': cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)})
                    clean_bbox = [class_id, float(x_c), float(y_c), float(bw), float(bh)]
                    obj_metadata[obj_id] = {
                        "area": area, "grad_D": grad_D, "surface_normal": surface_normal, "c_sh": c_sh,
                        "solidity": solidity, "source_image": img_path,
                        "bbox": clean_bbox, # <-- Save the clean version
                        "cluster": reg_info['cluster']
                    }
                    obj_counter += 1

# --- 7. PHASE B: DINOv2 BATCHING (RUNS ON CACHED PATCHES) ---
patch_loader = DataLoader(PatchDataset(patch_registry, dino_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
embeddings_dict = {}

with torch.no_grad():
    for batch_tensors, batch_ids in tqdm(patch_loader, desc="DINOv2 Object Profiling"):
        batch_tensors = batch_tensors.to(device)
        feats = dino(batch_tensors).cpu().numpy()
        for i, oid in enumerate(batch_ids):
            embeddings_dict[oid] = feats[i]

# --- 8. ISOLATION FOREST & JSON ASSEMBLY ---
latent_dictionary = {}
clusters = sorted(list(set([v['cluster'] for v in obj_metadata.values()])))

for c in tqdm(clusters, desc="Applying Isolation Sieve"):
    # Safe extraction using .get()
    c_objs = {k: v for k, v in obj_metadata.items() if v.get('cluster') == c}
    obj_ids = list(c_objs.keys())

    if len(obj_ids) > 10:
        c_embeds = np.vstack([embeddings_dict[oid] for oid in obj_ids])
        iso_forest = IsolationForest(contamination=0.10, random_state=SEED)
        labels = iso_forest.fit_predict(c_embeds)
        for idx, lbl in enumerate(labels):
            c_objs[obj_ids[idx]]["semantic_loss"] = bool(lbl == -1)
    else:
        for oid in obj_ids: c_objs[oid]["semantic_loss"] = False

    areas = [v['area'] for v in c_objs.values()]
    a_min = float(np.percentile(areas, 5)) if areas else 0.0
    a_max = float(np.percentile(areas, 95)) if areas else 1.0

    # 🚨 FIX: Safely strip utility tag using a deep copy to prevent KeyError on next iteration
    final_c_objs = {}
    for k, v in c_objs.items():
        v_copy = v.copy()
        v_copy.pop('cluster', None)
        final_c_objs[k] = v_copy

    latent_dictionary[c] = {"scale_bounds": {"A_min": a_min, "A_max": a_max}, "objects": final_c_objs}

with open(DICT_PATH, 'w') as f: json.dump(latent_dictionary, f, indent=4)

# 🔥 GPU VRAM Cleanup
del midas, dino, env_loader, patch_loader, depth_preds
torch.cuda.empty_cache()
gc.collect()

print(f"\n✅ Dictionary Optimized & Compiled! Synchronized {obj_counter} object constraints to {DICT_PATH}")

--- 🏗️ BUILDING CONTEXT-SYNCHRONIZED LATENT DICTIONARY ---
✅ Local Root Synchronized at: /content/IMAGE_WORKSPACE/COD10K-Structured-OriginalFormat
The repository intel-isl_MiDaS does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None
The repository rwightman_gen-efficientnet-pytorch does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt



  0%|          | 0.00/81.8M [00:00<?, ?B/s]
 37%|███▋      | 30.1M/81.8M [00:00<00:00, 297MB/s]
 72%|███████▏  | 58.5M/81.8M [00:00<00:00, 198MB/s]
100%|██████████| 81.8M/81.8M [00:00<00:00, 194MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main

Topography Extraction: 100%|██████████| 12/12 [06:35<00:00, 32.96s/it]

DINOv2 Object Profiling: 100%|██████████| 14/14 [00:05<00:00,  2.76it/s]

Applying Isolation Sieve: 100%|██████████| 25/25 [00:04<00:00,  5.83it/s]



✅ Dictionary Optimized & Compiled! Synchronized 3532 object constraints to /content/drive/MyDrive/CAMO/latent_affordance_dictionary.json


In [ ]:
# ==============================================================================
# CELL 7: UNIVERSAL BACKGROUND INGESTION (HYBRID APPROACH)
# ==============================================================================
import os
import cv2
import json
import torch
import numpy as np
import shutil
from glob import glob
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

print("--- 🌍 MULTI-THREADED UNIVERSAL BACKGROUND INGESTION ---")

sample_path = glob("/content/IMAGE_WORKSPACE/**/train", recursive=True)[0]
INPUT_YOLO_DIR = os.path.dirname(sample_path)

# Fast local paths for I/O bound tasks
RECLAIMED_SOURCE = "/content/CSP_Partitioned_Dataset/images"
CATALOG_PATH = "/content/CSP_Partitioned_Dataset/environment_catalog.json"
EXTERNAL_SOURCE = os.path.join(INPUT_YOLO_DIR, "train", "non-camo", "image")

RHO_MAX = 0.5
DINO_GRID_SIZE = 224 # Strictly for DINOv2 semantics, not spatial math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device).eval()

dino_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((DINO_GRID_SIZE, DINO_GRID_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Loading the pre-computed Macro-Environment Centroids
print("\n📥 Loading pre-computed Macro-Environment Centroids...")
CENTROID_PATH = "/content/drive/MyDrive/CAMO/habitat_centroids.npy"
try:
    habitat_centroids = np.load(CENTROID_PATH, allow_pickle=True).item()
    print(f"✅ Loaded anchors for {len(habitat_centroids)} distinct habitats.")
except FileNotFoundError:
    raise FileNotFoundError("❌ Cannot find habitat_centroids.npy!")

background_catalog = {}
all_image_paths = sorted(
    glob(os.path.join(RECLAIMED_SOURCE, "**", "*.jpg"), recursive=True) + \
    glob(os.path.join(EXTERNAL_SOURCE, "*.jpg"))
)
# --- 1. Multi-Threaded Dataset Definition ---
class BackgroundDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            return torch.zeros((3, DINO_GRID_SIZE, DINO_GRID_SIZE)), img_path, False, 0, 0
        h, w = img.shape[:2] # Capture NATIVE dimensions
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Transform yields a 224x224 tensor for DINOv2
        tensor = self.transform(img_rgb)

        # We return both the squished tensor AND the native dimensions
        return tensor, img_path, True, h, w

dataset = BackgroundDataset(all_image_paths, dino_transforms)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# --- 2. Batched Execution Loop ---
with torch.no_grad():
    for batch_tensors, batch_paths, batch_valid, batch_h, batch_w in tqdm(dataloader, desc="Cataloging Backgrounds"):

        # ---------------------------------------------------------
        # AI PART: Run DINOv2 on the 224x224 tensors (GPU)
        # ---------------------------------------------------------
        needs_dino_idx = []
        dino_tensors = []

        for i, path in enumerate(batch_paths):
            if not batch_valid[i]: continue
            if "CSP_Partitioned_Dataset" not in path:
                needs_dino_idx.append(i)
                dino_tensors.append(batch_tensors[i])

        batch_feats = {}
        if dino_tensors:
            dino_input = torch.stack(dino_tensors).to(device)
            dino_output = dino(dino_input).cpu().numpy()
            for out_idx, orig_idx in enumerate(needs_dino_idx):
                batch_feats[orig_idx] = dino_output[out_idx]

        # ---------------------------------------------------------
        # MATH PART: Process Masks natively (CPU)
        # ---------------------------------------------------------
        for i in range(len(batch_paths)):
            if not batch_valid[i]: continue

            img_path = batch_paths[i]
            img_name = os.path.basename(img_path)
            bg_base_name = os.path.splitext(img_name)[0]

            # 🚨 HYBRID LOGIC: Extract Native Dimensions
            orig_H, orig_W = batch_h[i].item(), batch_w[i].item()

            if "non-camo" in img_path:
                lbl_path = os.path.join(INPUT_YOLO_DIR, "train", "non-camo", "label", f"{bg_base_name}.txt")
            else:
                if "CSP_Partitioned_Dataset" in img_path:
                    lbl_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
                else:
                    lbl_path = os.path.join(INPUT_YOLO_DIR, "train", "camo", "label", f"{bg_base_name}.txt")

            # 🚨 HYBRID LOGIC: Create a mask exactly the size of the high-res image
            existing_mask = np.zeros((orig_H, orig_W), dtype=np.uint8)

            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    for line in f.readlines():
                        try:
                            bbox = [float(x) for x in line.strip().split()][:5]
                            _, x_c, y_c, bw, bh = bbox
                            # 🚨 HYBRID LOGIC: Multiply YOLO percentages by NATIVE dimensions
                            x1, x2 = int((x_c - bw/2) * orig_W), int((x_c + bw/2) * orig_W)
                            y1, y2 = int((y_c - bh/2) * orig_H), int((y_c + bh/2) * orig_H)
                            cv2.rectangle(existing_mask, (x1, y1), (x2, y2), 255, -1)
                        except Exception: continue

            # Calculate saturation accurately based on native pixel count
            saturation = np.count_nonzero(existing_mask) / (orig_H * orig_W)
            if saturation >= RHO_MAX: continue

            # 🚨 HYBRID LOGIC: Distance transform runs on the high-res mask
            empty_space = np.uint8((existing_mask == 0) * 255)
            # Using standard Euclidean distance (DIST_L2) with a 5x5 mask size
            dist_transform = cv2.distanceTransform(empty_space, cv2.DIST_L2, 5)

            # Normalize based on the native width
            normalized_r_avail = float(np.max(dist_transform)) / orig_W

            if normalized_r_avail < 0.02: continue

            # ---------------------------------------------------------
            # CLASSIFICATION PART: Link it together
            # ---------------------------------------------------------
            best_match_cluster = None
            best_sim = 1.0

            if "CSP_Partitioned_Dataset" in img_path:
                best_match_cluster = os.path.basename(os.path.dirname(img_path))
            else:
                bg_feat = batch_feats[i]
                best_sim = -1
                for cluster_name, centroid in habitat_centroids.items():
                    sim = cosine_similarity(bg_feat.reshape(1, -1), centroid.reshape(1, -1))[0][0]
                    if sim > best_sim: best_sim, best_match_cluster = sim, cluster_name
                if best_sim < 0.65: continue

            if best_match_cluster not in background_catalog: background_catalog[best_match_cluster] = {}
            background_catalog[best_match_cluster][img_name] = {
                "path": img_path,
                "semantic_confidence": float(best_sim),
                "saturation": float(saturation),
                "R_avail_norm": float(normalized_r_avail),
                "orig_w": orig_W,
                "orig_h": orig_H
            }

with open(CATALOG_PATH, 'w') as f: json.dump(background_catalog, f, indent=4)
print(f"\n✅ Multi-Threaded Hybrid Ingestion Map Cataloged to {CATALOG_PATH}.")

# Optional: Back it up to Drive immediately since it's just a text file
shutil.copy(CATALOG_PATH, "/content/drive/MyDrive/CAMO/")
print("💾 Backup saved to Google Drive.")

--- 🌍 MULTI-THREADED UNIVERSAL BACKGROUND INGESTION ---


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



📥 Loading pre-computed Macro-Environment Centroids...
✅ Loaded anchors for 25 distinct habitats.



Cataloging Backgrounds: 100%|██████████| 24/24 [01:11<00:00,  2.99s/it]



✅ Multi-Threaded Hybrid Ingestion Map Cataloged to /content/CSP_Partitioned_Dataset/environment_catalog.json.
💾 Backup saved to Google Drive.


In [ ]:
# ==============================================================================
# CELL 8: MASTER SYNTHESIS ENGINE (A100 SCALED & TASK-CENTRIC)
# ==============================================================================
import os
import cv2
import json
import random
import torch
import numpy as np
import gc
import shutil
import subprocess
from glob import glob
from tqdm import tqdm
from scipy.spatial.distance import cosine
from skimage.feature import local_binary_pattern
from torch.utils.data import Dataset, DataLoader

print("--- 🧬 EXECUTING GPU-ACCELERATED SYNTHESIS ENGINE ---")

# --- 1. A100 Optimized Hardware Settings ---
BATCH_SIZE = 512
NUM_WORKERS = 12
MAX_WORKERS = 32

MAX_RETRIES = 50
BLUR_THRESHOLD = 50.0
Z_BUFFER_MARGIN = 0.05
LBP_RADIUS = 3
LBP_POINTS = 8 * LBP_RADIUS
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fast Local SSD Paths
LOCAL_OUTPUT_DIR = "/content/Synthesized_Output"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/CAMO"
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

try:
    sample_path = glob("/content/IMAGE_WORKSPACE/**/train", recursive=True)[0]
    INPUT_YOLO_DIR = os.path.dirname(sample_path)
except IndexError:
    INPUT_YOLO_DIR = "/content/IMAGE_WORKSPACE"

# --- 2. Physics & Math Functions ---
def calculate_laplacian_variance(img_patch):
    gray = cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def get_patch_surface_normal(depth_patch):
    grad_x = cv2.Sobel(depth_patch, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(depth_patch, cv2.CV_64F, 0, 1, ksize=3)
    mean_gx, mean_gy = np.mean(grad_x), np.mean(grad_y)
    magnitude = np.sqrt(mean_gx**2 + mean_gy**2 + 1.0)
    raw_normal = [-mean_gx / magnitude, -mean_gy / magnitude, 1.0 / magnitude]
    n_mag = np.sqrt(sum(n**2 for n in raw_normal)) + 1e-8
    return [n / n_mag for n in raw_normal]

def get_patch_spherical_harmonics(img_patch, mask=None):
    gray = cv2.cvtColor(img_patch, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    if mask is not None and np.count_nonzero(mask) > 0:
        gray = gray * (mask.astype(np.float32) / 255.0)

    ph, pw = gray.shape
    y_grid, x_grid = np.mgrid[-1:1:ph*1j, -1:1:pw*1j]
    r2 = x_grid**2 + y_grid**2
    valid = r2 <= 1.0
    z_grid = np.zeros_like(x_grid)
    z_grid[valid] = np.sqrt(1 - r2[valid])
    Y = [
        np.ones_like(x_grid) * 0.282095, y_grid * 0.488603, z_grid * 0.488603, x_grid * 0.488603,
        (x_grid * y_grid) * 1.092548, (y_grid * z_grid) * 1.092548,
        (3.0 * z_grid**2 - 1.0) * 0.315392, (x_grid * z_grid) * 1.092548, (x_grid**2 - y_grid**2) * 0.546274
    ]
    sum_valid = np.sum(valid)
    if sum_valid == 0: return [0.0] * 9
    return [float(np.sum(gray[valid] * basis[valid]) / sum_valid) for basis in Y]

def calculate_lbp_distance(patch_a, patch_b):
    gray_a = cv2.cvtColor(patch_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(patch_b, cv2.COLOR_BGR2GRAY)
    lbp_a = local_binary_pattern(gray_a, LBP_POINTS, LBP_RADIUS, method="uniform")
    lbp_b = local_binary_pattern(gray_b, LBP_POINTS, LBP_RADIUS, method="uniform")
    hist_a, _ = np.histogram(lbp_a.ravel(), bins=np.arange(0, LBP_POINTS + 3), range=(0, LBP_POINTS + 2))
    hist_b, _ = np.histogram(lbp_b.ravel(), bins=np.arange(0, LBP_POINTS + 3), range=(0, LBP_POINTS + 2))

    # Safe division to prevent RuntimeWarning
    sum_a = hist_a.sum()
    sum_b = hist_b.sum()
    hist_a = hist_a.astype("float") / (sum_a if sum_a > 0 else 1.0)
    hist_b = hist_b.astype("float") / (sum_b if sum_b > 0 else 1.0)

    return np.linalg.norm(hist_a - hist_b)

def apply_z_buffer_masking(obj_mask, bg_depth_patch, obj_base_depth):
    occlusion_mask = (bg_depth_patch > (obj_base_depth + Z_BUFFER_MARGIN)).astype(np.uint8) * 255
    return cv2.bitwise_and(obj_mask, cv2.bitwise_not(occlusion_mask))

def transfer_color(source_patch, target_patch, mask):
    src_lab = cv2.cvtColor(source_patch, cv2.COLOR_BGR2LAB).astype(np.float32)
    tgt_lab = cv2.cvtColor(target_patch, cv2.COLOR_BGR2LAB).astype(np.float32)
    mask_bool = mask > 127
    if not np.any(mask_bool): return source_patch, 0.0

    src_mean, src_std = cv2.meanStdDev(src_lab, mask=mask)
    tgt_mean, tgt_std = cv2.meanStdDev(tgt_lab)
    shift_magnitude = np.sqrt(sum((src_mean[i][0] - tgt_mean[i][0])**2 for i in range(3)))
    src_std[src_std == 0] = 1e-5

    result_lab = np.empty_like(src_lab)
    result_lab[:, :, 0] = (src_lab[:, :, 0] - src_mean[0][0]) + tgt_mean[0][0]

    for i in range(1, 3):
        std_ratio = np.clip(tgt_std[i][0] / src_std[i][0], 0.5, 1.5)
        result_lab[:, :, i] = ((src_lab[:, :, i] - src_mean[i][0]) * std_ratio) + tgt_mean[i][0]

    result_lab = np.clip(result_lab, 0, 255).astype(np.uint8)
    final_source = source_patch.copy()
    final_source[mask_bool] = cv2.cvtColor(result_lab, cv2.COLOR_LAB2BGR)[mask_bool]
    return final_source, shift_magnitude

# --- 3. Initializing Models & Loading Dictionaries ---
print("Loading MiDaS Core...")
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device).eval()
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

with open("/content/CSP_Partitioned_Dataset/environment_catalog.json", 'r') as f: environments = json.load(f)
with open("/content/drive/MyDrive/CAMO/latent_affordance_dictionary.json", 'r') as f: dictionary = json.load(f)

# --- 4. Task-Centric Flattener (Preventing Stragglers) ---
flattened_tasks = []

for env_name, env_data in environments.items():
    if isinstance(env_data, dict) and "path" in env_data:
        items = [(env_name, env_data)]
    else:
        items = env_data.items()

    for item_key, item_meta in items:
        habitat = env_name if not isinstance(env_data, dict) or "path" not in env_data else env_data.get("matched_habitat", "global_habitat")
        bg_path = item_meta["path"]

        if habitat in dictionary and dictionary[habitat]["objects"]:
            habitat_objects = list(dictionary[habitat]["objects"].values())

            for obj in habitat_objects:
                # Pre-filter doomed objects to save CPU cycles
                if dictionary[habitat]["scale_bounds"]["A_min"] <= obj["area"] <= dictionary[habitat]["scale_bounds"]["A_max"]:
                    flattened_tasks.append({
                        "bg_path": bg_path,
                        "obj_data": obj,
                        "habitat": habitat
                    })

print(f"📊 Flattened into {len(flattened_tasks)} individual atomic synthesis tasks.")

# --- 5. Atomic Dataloader ---
class AtomicSynthesisDataset(Dataset):
    def __init__(self, tasks, transform):
        self.tasks = tasks
        self.transform = transform

    def __len__(self): return len(self.tasks)

    def __getitem__(self, idx):
        task = self.tasks[idx]
        bg_path = task["bg_path"]

        img = cv2.imread(bg_path)
        if img is None:
            return torch.zeros(3, 256, 256), bg_path, json.dumps({}), False, 0, 0

        h, w = img.shape[:2]
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (256, 256))
        tensor = self.transform(img_rgb).squeeze(0)

        # Serialize dict to string for dataloader safety
        obj_json = json.dumps(task["obj_data"])
        return tensor, bg_path, obj_json, True, h, w

dataloader = DataLoader(
    AtomicSynthesisDataset(flattened_tasks, midas_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True, # Crucial for load balancing across habitats
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# --- 6. The Synthesis Execution Loop ---
success_count = 0
source_image_cache = {}
print(f"🚀 Ready: Processing tasks via GPU Batching.")

with torch.no_grad():
    for batch_tensors, batch_paths, batch_obj_jsons, batch_valid, batch_h, batch_w in tqdm(dataloader, desc="Synthesizing Scenes"):
        batch_tensors = batch_tensors.to(device)
        depth_preds = midas(batch_tensors)

        for i in range(len(batch_paths)):
            if not batch_valid[i]: continue

            bg_path = batch_paths[i]
            obj = json.loads(batch_obj_jsons[i]) # Deserialize object

            H, W = batch_h[i].item(), batch_w[i].item()
            bg_base_name = os.path.splitext(os.path.basename(bg_path))[0]

            bg_depth = torch.nn.functional.interpolate(
                depth_preds[i].unsqueeze(0).unsqueeze(0), size=(H, W), mode="bicubic", align_corners=False
            ).squeeze().cpu().numpy()
            bg_depth = (bg_depth - bg_depth.min()) / (bg_depth.max() - bg_depth.min() + 1e-8)

            bg_img = cv2.imread(bg_path)

            # Extract Background Masks & Labels
            bg_mask = np.zeros((H, W), dtype=np.uint8)
            possible_bg_masks = glob(os.path.join(INPUT_YOLO_DIR, "**", "mask", f"{bg_base_name}.*"), recursive=True)
            if possible_bg_masks:
                loaded_bg_mask = cv2.imread(possible_bg_masks[0], cv2.IMREAD_GRAYSCALE)
                if loaded_bg_mask is not None:
                    _, bg_mask = cv2.threshold(loaded_bg_mask, 127, 255, cv2.THRESH_BINARY)
                    if bg_mask.shape[:2] != (H, W): bg_mask = cv2.resize(bg_mask, (W, H), interpolation=cv2.INTER_NEAREST)

            existing_bg_labels = []
            possible_bg_labels = glob(os.path.join(INPUT_YOLO_DIR, "**", "label", f"{bg_base_name}.txt"), recursive=True)
            if possible_bg_labels:
                with open(possible_bg_labels[0], 'r') as f: existing_bg_labels = f.readlines()

            # --- START SINGLE OBJECT PROCESSING ---
            _, src_xc, src_yc, src_bw, src_bh = obj["bbox"]
            obj_base_name = os.path.splitext(os.path.basename(obj["source_image"]))[0]

            original_class_id = "-"
            possible_obj_labels = glob(os.path.join(INPUT_YOLO_DIR, "**", "label", f"{obj_base_name}.txt"), recursive=True)

            if possible_obj_labels:
                with open(possible_obj_labels[0], 'r') as f_label:
                    for line in f_label:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            try:
                                lbl_xc, lbl_yc = float(parts[1]), float(parts[2])
                                if abs(lbl_xc - src_xc) < 1e-3 and abs(lbl_yc - src_yc) < 1e-3:
                                    original_class_id = str(int(float(parts[0])))
                                    break
                            except ValueError: pass

            possible_obj_masks = glob(os.path.join(INPUT_YOLO_DIR, "**", "mask", f"{obj_base_name}.*"), recursive=True)
            if not possible_obj_masks: continue

            obj_src_mask = cv2.imread(possible_obj_masks[0], cv2.IMREAD_GRAYSCALE)
            if obj_src_mask is None: continue

            # A100 RAM Cache (Hold up to 500 images)
            src_path = obj["source_image"]
            if src_path not in source_image_cache:
                img_load = cv2.imread(src_path)
                if img_load is None: continue
                source_image_cache[src_path] = img_load
                if len(source_image_cache) > 500:
                    source_image_cache.pop(next(iter(source_image_cache)))

            src_img = source_image_cache[src_path]
            src_H, src_W = src_img.shape[:2]

            # Try to place the object up to MAX_RETRIES times
            placement_successful = False

            for attempt in range(MAX_RETRIES):
                pad_x_pct = 0.05
                pad_y_pct = 0.05
                padded_bw = src_bw + (src_bw * pad_x_pct * 2)
                padded_bh = src_bh + (src_bh * pad_y_pct * 2)

                target_w, target_h = int(padded_bw * W), int(padded_bh * H)
                if target_w >= W or target_h >= H or target_w <= 10 or target_h <= 10: break

                prop_x = random.randint(5, max(5, W - target_w - 5))
                prop_y = random.randint(5, max(5, H - target_h - 5))

                bg_patch = bg_img[prop_y:prop_y+target_h, prop_x:prop_x+target_w]
                actual_h, actual_w = bg_patch.shape[:2]
                if actual_h <= 5 or actual_w <= 5: continue

                bg_mask_patch = bg_mask[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w]
                if np.any(bg_mask_patch > 0): continue
                bg_depth_patch = bg_depth[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w]

                x_min = int((src_xc - src_bw / 2.0) * src_W)
                x_max = int((src_xc + src_bw / 2.0) * src_W)
                y_min = int((src_yc - src_bh / 2.0) * src_H)
                y_max = int((src_yc + src_bh / 2.0) * src_H)

                p_x = int(src_bw * pad_x_pct * src_W)
                p_y = int(src_bh * pad_y_pct * src_H)

                sx1, sx2 = max(0, x_min - p_x), min(src_W, x_max + p_x)
                sy1, sy2 = max(0, y_min - p_y), min(src_H, y_max + p_y)
                if (sx2 - sx1) < (x_max - x_min) or (sy2 - sy1) < (y_max - y_min): break

                raw_obj_crop = src_img[sy1:sy2, sx1:sx2]
                if raw_obj_crop.size == 0: break

                # 50% chance to horizontally flip object
                if random.random() > 0.5:
                    raw_obj_crop = cv2.flip(raw_obj_crop, 1)
                    is_flipped = True
                else:
                    is_flipped = False

                if obj_src_mask.shape[:2] != (src_H, src_W):
                    resized_mask = cv2.resize(obj_src_mask, (src_W, src_H), interpolation=cv2.INTER_NEAREST)
                else: resized_mask = obj_src_mask

                raw_mask_crop = resized_mask[sy1:sy2, sx1:sx2]
                if is_flipped: raw_mask_crop = cv2.flip(raw_mask_crop, 1)

                obj_mask = cv2.resize(raw_mask_crop, (actual_w, actual_h), interpolation=cv2.INTER_NEAREST)
                _, obj_mask = cv2.threshold(obj_mask, 127, 255, cv2.THRESH_BINARY)

                num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(obj_mask, connectivity=8)
                if num_labels > 2:
                    center_x, center_y = actual_w // 2, actual_h // 2
                    best_label = 1
                    min_dist = float('inf')
                    for l in range(1, num_labels):
                        cx, cy = centroids[l]
                        dist = (cx - center_x)**2 + (cy - center_y)**2
                        if dist < min_dist:
                            min_dist = dist
                            best_label = l
                    obj_mask = np.where(labels == best_label, 255, 0).astype(np.uint8)

                if obj.get("semantic_loss", False):
                    obj_mask = apply_z_buffer_masking(obj_mask, bg_depth_patch, np.median(bg_depth_patch))
                    if np.count_nonzero(obj_mask) < (actual_w * actual_h * 0.2): continue

                if calculate_laplacian_variance(bg_patch) < BLUR_THRESHOLD: continue
                if np.dot(obj["surface_normal"], get_patch_surface_normal(bg_depth_patch)) < 0.75: continue

                # Use scipy cosine safely
                sh_bg = get_patch_spherical_harmonics(bg_patch, mask=obj_mask)
                sh_obj = obj["c_sh"]

                # Check for zero vectors before calculating cosine
                if np.sum(np.abs(sh_bg)) == 0 or np.sum(np.abs(sh_obj)) == 0: continue

                try:
                    if (1.0 - cosine(sh_obj, sh_bg)) < 0.80: continue
                except Exception: continue

                obj_pixels = cv2.resize(raw_obj_crop, (actual_w, actual_h))
                color_matched_obj, shift_mag = transfer_color(obj_pixels, bg_patch, obj_mask)
                if shift_mag > 40.0: continue

                isolated_obj_pixels = cv2.bitwise_and(obj_pixels, obj_pixels, mask=obj_mask)
                if calculate_lbp_distance(isolated_obj_pixels, bg_patch) > (0.6 * obj.get("solidity", 0.50)): continue

                pristine_obj_mask = obj_mask.copy()
                temp_bg = bg_img.copy()

                erosion_kernel = np.ones((5, 5), np.uint8)
                blend_mask = cv2.erode(obj_mask, erosion_kernel, iterations=1)
                _, hard_binary_mask = cv2.threshold(blend_mask, 127, 255, cv2.THRESH_BINARY)
                if np.count_nonzero(hard_binary_mask) < 10: continue

                center_coordinate = (prop_x + actual_w // 2, prop_y + actual_h // 2)

                try:
                    harmonized_full_canvas = cv2.seamlessClone(
                        color_matched_obj, temp_bg.copy(), hard_binary_mask, center_coordinate, cv2.NORMAL_CLONE
                    )
                    harmonized_patch = harmonized_full_canvas[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w]
                except Exception: continue

                alpha_mask = cv2.GaussianBlur(pristine_obj_mask, (3, 3), 0).astype(np.float32) / 255.0
                if len(alpha_mask.shape) == 2: alpha_mask = np.expand_dims(alpha_mask, axis=2)

                bg_patch_float = temp_bg[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w].astype(np.float32)
                harmonized_float = harmonized_patch.astype(np.float32)
                final_hybrid_patch = (harmonized_float * alpha_mask) + (bg_patch_float * (1.0 - alpha_mask))
                final_hybrid_patch = np.clip(final_hybrid_patch, 0, 255).astype(np.uint8)
                temp_bg[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w] = final_hybrid_patch

                full_canvas_obj_mask = np.zeros((H, W), dtype=np.uint8)
                full_canvas_obj_mask[prop_y:prop_y+actual_h, prop_x:prop_x+actual_w] = pristine_obj_mask
                final_merged_mask = cv2.bitwise_or(bg_mask, full_canvas_obj_mask)

                mask_contours, _ = cv2.findContours(pristine_obj_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if mask_contours:
                    all_pts = np.vstack(mask_contours)
                    x_b, y_b, w_b, h_b = cv2.boundingRect(all_pts)
                    tight_xc = prop_x + x_b + w_b / 2.0
                    tight_yc = prop_y + y_b + h_b / 2.0
                    tight_w = float(w_b)
                    tight_h = float(h_b)
                else:
                    tight_xc = prop_x + actual_w / 2.0
                    tight_yc = prop_y + actual_h / 2.0
                    tight_w = float(actual_w)
                    tight_h = float(actual_h)

                new_bbox_line = f"{original_class_id} {tight_xc / W:.6f} {tight_yc / H:.6f} {tight_w / W:.6f} {tight_h / H:.6f}\n"

                final_labels = existing_bg_labels.copy()
                final_labels.append(new_bbox_line)

                # Write to Fast Local SSD
                cv2.imwrite(os.path.join(LOCAL_OUTPUT_DIR, f"synth_{success_count}.jpg"), temp_bg)
                cv2.imwrite(os.path.join(LOCAL_OUTPUT_DIR, f"synth_{success_count}.png"), final_merged_mask)
                with open(os.path.join(LOCAL_OUTPUT_DIR, f"synth_{success_count}.txt"), 'w') as f:
                    f.writelines(final_labels)

                success_count += 1
                placement_successful = True
                break # Exit the retry loop for this object

            # Note: Because this is an atomic task, we are done with this iteration regardless of success.

# --- 7. Final Cleanup & Export ---
del midas, dataloader, depth_preds
torch.cuda.empty_cache()
gc.collect()

print(f"\n✅ Generation Loop Complete. ({success_count} images synthesized locally)")
print("📦 Zipping synthesized dataset for Google Drive transfer... This may take a moment.")

ZIP_NAME = "/content/Synthesized_Output.zip"
if os.path.exists(ZIP_NAME):
    os.remove(ZIP_NAME)

subprocess.run(["zip", "-r", "-q", ZIP_NAME, LOCAL_OUTPUT_DIR])

print(f"🚀 Moving {ZIP_NAME} to Google Drive ({DRIVE_BACKUP_DIR})...")
shutil.copy(ZIP_NAME, DRIVE_BACKUP_DIR)

print(f"✅ Master Synthesis Pipeline Complete! Synthetic dataset is safely backed up to Drive.")

--- 🧬 EXECUTING GPU-ACCELERATED SYNTHESIS ENGINE ---
Loading MiDaS Core...


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Loading weights:  None


Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


📊 Flattened into 492888 individual atomic synthesis tasks.
🚀 Ready: Processing tasks via GPU Batching.



Synthesizing Scenes:   0%|          | 0/963 [00:00<?, ?it/s]

In [ ]:
# ==============================================================================
# CELL 9: UNIFIED DATASET REORGANIZER & PACKAGER (ORIGINAL FORMAT)
# ==============================================================================
import os
import shutil
from glob import glob
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

print("--- 📦 PACKAGING UNIFIED DATASET (TARGET STRUCTURE) ---")

# --- 1. Path Configuration ---
SYNTHESIZED_DIR = "/content/Synthesized_Output"
STAGING_DIR = "/content/YOLO_Export"
ZIP_OUTPUT_PATH = "/content/drive/MyDrive/CAMO/COD10K-AUG-OURS"
MAX_WORKERS = 8

# Locate the root of your original dataset workspace
try:
    sample_path = glob("/content/IMAGE_WORKSPACE/**/train", recursive=True)[0]
    INPUT_YOLO_DIR = os.path.dirname(sample_path)
except IndexError:
    INPUT_YOLO_DIR = "/content/IMAGE_WORKSPACE"

# Clean staging area if re-running
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR, exist_ok=True)

# --- 2. Build Target Directory Structure ---
train_img_dir = os.path.join(STAGING_DIR, "train", "image")
train_lbl_dir = os.path.join(STAGING_DIR, "train", "label")
train_msk_dir = os.path.join(STAGING_DIR, "train", "mask")

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)
os.makedirs(train_msk_dir, exist_ok=True)

# --- 3. Exact Copy for Val and Data.yaml ---
print("📂 Copying 'val' directory and 'data.yaml' directly from source...")
val_src = os.path.join(INPUT_YOLO_DIR, "val")
if os.path.exists(val_src):
    shutil.copytree(val_src, os.path.join(STAGING_DIR, "val"))
else:
    print("⚠️ Warning: 'val' directory not found in source.")

yaml_src = os.path.join(INPUT_YOLO_DIR, "data.yaml")
if os.path.exists(yaml_src):
    shutil.copy(yaml_src, os.path.join(STAGING_DIR, "data.yaml"))

# --- 4. Define Fast Copy Task ---
def copy_file_task(src_path, dest_path):
    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)

tasks = []

# --- 5. Queue Original Train Data (Merge camo & non-camo) ---
print("🔍 Queuing original 'camo' and 'non-camo' training files...")
for subset in ["camo", "non-camo"]:
    for folder_type in ["image", "label", "mask"]:
        src_folder = os.path.join(INPUT_YOLO_DIR, "train", subset, folder_type)
        dest_folder = os.path.join(STAGING_DIR, "train", folder_type)

        if os.path.exists(src_folder):
            for filename in os.listdir(src_folder):
                src_file = os.path.join(src_folder, filename)
                dest_file = os.path.join(dest_folder, filename)
                tasks.append((src_file, dest_file))

# --- 6. Queue Synthesized Data (Split by extension) ---
print("🧪 Queuing newly synthesized dataset...")
if os.path.exists(SYNTHESIZED_DIR):
    for filename in os.listdir(SYNTHESIZED_DIR):
        src_file = os.path.join(SYNTHESIZED_DIR, filename)

        # Route based on extension
        if filename.lower().endswith(('.jpg', '.jpeg')):
            dest_file = os.path.join(train_img_dir, filename)
        elif filename.lower().endswith('.txt'):
            dest_file = os.path.join(train_lbl_dir, filename)
        elif filename.lower().endswith('.png'):
            dest_file = os.path.join(train_msk_dir, filename)
        else:
            continue

        tasks.append((src_file, dest_file))

# --- 7. Execute High-Speed Multithreaded Routing ---
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(copy_file_task, src, dest) for src, dest in tasks]

    for _ in tqdm(as_completed(futures), total=len(futures), desc="Merging & Routing Files"):
        pass

# --- 8. Final Zip and Export ---
print(f"\n📦 Dataset constructed. Zipping to Google Drive...")
# shutil.make_archive automatically appends '.zip' to the output path
shutil.make_archive(ZIP_OUTPUT_PATH, 'zip', STAGING_DIR)

print(f"✅ EXPORT COMPLETE! Unified dataset saved to {ZIP_OUTPUT_PATH}.zip")